In [4]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 16.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [5]:
!python -c "import nltk; nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('stopwords'); nltk.download('vader_lexicon'); nltk.download('popular')"

[nltk_data] Downloading package punkt to /home/ayush/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/ayush/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /home/ayush/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/ayush/nltk_data...
[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /home/ayush/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /home/ayush/nltk_data...
[nltk_data]    |   Unzipping corpora/gazetteers.zip.
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /home/ayush/nltk_data...
[nltk_data]    |   Unzipping corpora/genesis.zip.
[nltk_data]    | Downloading package gutenberg to
[nltk_data

In [6]:
import os
import json
from pathlib import Path

def create_project_structure():
    """Create the recommended project directory structure"""

    directories = [
        'data/raw/liar',
        'data/raw/fakenewsnet',
        'data/raw/indian_context',
        'data/raw/sentiment_analysis',
        'data/processed',
        'data/external',
        'models',
        'notebooks',
        'src/data_collection',
        'src/preprocessing',
        'src/models',
        'src/evaluation',
        'configs',
        'logs',
        'scripts'
    ]

    for directory in directories:
        Path(directory).mkdir(parents=True, exist_ok=True)
        print(f"Created directory: {directory}")

    # Create empty __init__.py files for Python packages
    init_dirs = ['src', 'src/data_collection', 'src/preprocessing', 'src/models', 'src/evaluation']
    for directory in init_dirs:
        Path(f"{directory}/__init__.py").touch()

    print("Project structure created successfully!")

# Execute the function
create_project_structure()

Created directory: data/raw/liar
Created directory: data/raw/fakenewsnet
Created directory: data/raw/indian_context
Created directory: data/raw/sentiment_analysis
Created directory: data/processed
Created directory: data/external
Created directory: models
Created directory: notebooks
Created directory: src/data_collection
Created directory: src/preprocessing
Created directory: src/models
Created directory: src/evaluation
Created directory: configs
Created directory: logs
Created directory: scripts
Project structure created successfully!


In [7]:
config_content = '''
import os
from pathlib import Path

# Project paths
PROJECT_ROOT = Path(__file__).parent.parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
LOGS_DIR = PROJECT_ROOT / "logs"

# Dataset paths
LIAR_DATA_PATH = RAW_DATA_DIR / "liar"
FAKENEWSNET_DATA_PATH = RAW_DATA_DIR / "fakenewsnet"
INDIAN_CONTEXT_DATA_PATH = RAW_DATA_DIR / "indian_context"
SENTIMENT_DATA_PATH = RAW_DATA_DIR / "sentiment_analysis"

# Model configurations
MODEL_CONFIGS = {
    "distilbert": {
        "model_name": "distilbert-base-uncased",
        "max_length": 512,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "num_epochs": 3
    },
    "roberta": {
        "model_name": "roberta-base",
        "max_length": 512,
        "batch_size": 16,
        "learning_rate": 1e-5,
        "num_epochs": 3
    }
}

# Scraping configurations
SCRAPING_CONFIG = {
    "user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "delay": 1,  # seconds between requests
    "timeout": 30,
    "max_retries": 3
}

# Indian news sources
RELIABLE_INDIAN_SOURCES = [
    "https://www.thehindu.com",
    "https://indianexpress.com",
    "https://www.hindustantimes.com",
    "https://timesofindia.indiatimes.com",
    "https://www.ndtv.com",
    "https://www.business-standard.com",
    "https://economictimes.indiatimes.com"
]

FACT_CHECK_SOURCES = [
    "https://www.altnews.in",
    "https://www.boomlive.in",
    "https://factly.in",
    "https://newsmobile.in/articles/category/fake-news/"
]

# API Keys (set in environment variables)
TWITTER_BEARER_TOKEN = os.getenv("TWITTER_BEARER_TOKEN")
NEWS_API_KEY = os.getenv("NEWS_API_KEY")
'''

with open('configs/config.py', 'w') as f:
    f.write(config_content)

print("Configuration file created: configs/config.py")

Configuration file created: configs/config.py


In [8]:
import requests
import pandas as pd
from datasets import load_dataset
import os
from pathlib import Path

def download_liar_dataset():
    """Download and setup LIAR dataset from HuggingFace"""

    print("Downloading LIAR dataset...")

    try:
        # Using HuggingFace datasets library
        dataset = load_dataset("ucsbnlp/liar")

        # Create directories
        liar_path = Path("data/raw/liar")
        liar_path.mkdir(parents=True, exist_ok=True)

        # Save each split as CSV
        splits = ['train', 'validation', 'test']

        for split in splits:
            if split in dataset:
                df = pd.DataFrame(dataset[split])
                output_path = liar_path / f"liar_{split}.csv"
                df.to_csv(output_path, index=False)
                print(f"Saved {split} split: {output_path} ({len(df)} samples)")

        print("LIAR dataset downloaded successfully!")
        return True

    except Exception as e:
        print(f"Error downloading LIAR dataset: {e}")
        print("Attempting alternative download method...")

        # Alternative: Direct download from UCSB
        try:
            url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
            response = requests.get(url, stream=True)

            if response.status_code == 200:
                with open("data/raw/liar/liar_dataset.zip", "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

                # Extract the zip file
                import zipfile
                with zipfile.ZipFile("data/raw/liar/liar_dataset.zip", 'r') as zip_ref:
                    zip_ref.extractall("data/raw/liar/")

                print("LIAR dataset downloaded via alternative method!")
                return True
            else:
                print(f"Failed to download LIAR dataset. Status code: {response.status_code}")
                return False

        except Exception as e2:
            print(f"Alternative download also failed: {e2}")
            return False

# Execute download
download_liar_dataset()

/home/ayush/coding/ml_projects/fake-news/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error downloading LIAR dataset: Dataset scripts are no longer supported, but found liar.py
Attempting alternative download method...
LIAR dataset downloaded via alternative method!


True

In [9]:
import requests
import json
import csv
from pathlib import Path

def setup_fakenewsnet_dataset():
    """Setup FakeNewsNet dataset structure and download metadata"""

    print("Setting up FakeNewsNet dataset...")

    # Create directory structure
    fakenewsnet_path = Path("data/raw/fakenewsnet")
    fakenewsnet_path.mkdir(parents=True, exist_ok=True)

    # Download the CSV files from GitHub
    base_url = "https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset"

    files_to_download = [
        "politifact_fake.csv",
        "politifact_real.csv",
        "gossipcop_fake.csv",
        "gossipcop_real.csv"
    ]

    for filename in files_to_download:
        try:
            url = f"{base_url}/{filename}"
            response = requests.get(url)

            if response.status_code == 200:
                output_path = fakenewsnet_path / filename
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(response.text)

                # Load and display basic info
                df = pd.read_csv(output_path)
                print(f"Downloaded {filename}: {len(df)} articles")
            else:
                print(f"Failed to download {filename}: {response.status_code}")

        except Exception as e:
            print(f"Error downloading {filename}: {e}")

    print("FakeNewsNet dataset setup completed!")

# Execute setup
setup_fakenewsnet_dataset()

Setting up FakeNewsNet dataset...
Downloaded politifact_fake.csv: 432 articles
Downloaded politifact_real.csv: 624 articles
Downloaded gossipcop_fake.csv: 5323 articles
Downloaded gossipcop_real.csv: 16817 articles
FakeNewsNet dataset setup completed!


In [15]:
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import urljoin, urlparse
from newspaper import Article
import pandas as pd
from datetime import datetime, timedelta

class NewsScraper:
    """A robust news scraping utility"""

    def __init__(self, delay_range=(1, 3), timeout=30):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        self.delay_range = delay_range
        self.timeout = timeout

    def random_delay(self):
        """Add random delay between requests"""
        time.sleep(random.uniform(*self.delay_range))

    def get_page(self, url, max_retries=3):
        """Get webpage content with retries"""
        for attempt in range(max_retries):
            try:
                response = self.session.get(url, timeout=self.timeout)
                response.raise_for_status()
                return response
            except Exception as e:
                print(f"Attempt {attempt + 1} failed for {url}: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
        return None

    def extract_article_with_newspaper3k(self, url):
        """Extract article content using newspaper3k"""
        try:
            article = Article(url, language='en')
            article.download()
            article.parse()

            return {
                'url': url,
                'title': article.title,
                'text': article.text,
                'authors': article.authors,
                'publish_date': article.publish_date,
                'summary': article.summary if hasattr(article, 'summary') else '',
                'keywords': article.keywords if hasattr(article, 'keywords') else []
            }
        except Exception as e:
            print(f"Error extracting article from {url}: {e}")
            return None

    def extract_links_from_sitemap(self, sitemap_url):
        """Extract article URLs from sitemap"""
        try:
            response = self.get_page(sitemap_url)
            if not response:
                return []

            soup = BeautifulSoup(response.content, 'xml')
            urls = [loc.text for loc in soup.find_all('loc')]
            return urls
        except Exception as e:
            print(f"Error extracting sitemap {sitemap_url}: {e}")
            return []

# Initialize scraper
scraper = NewsScraper()
print("News scraper initialized successfully!")

News scraper initialized successfully!


In [ ]:
def scrape_altnews_articles(max_articles=500):
    """Scrape fact-checking articles from Alt News"""

    print("Scraping Alt News fact-checking articles...")

    scraped_articles = []
    base_url = "https://www.altnews.in"

    # Alt News article pages (you may need to adjust based on their current structure)
    search_urls = [
        f"{base_url}/page/{page}/" for page in range(1, 21)  # First 20 pages
    ]

    for search_url in search_urls:
        try:
            response = scraper.get_page(search_url)
            if not response:
                continue

            soup = BeautifulSoup(response.content, 'html.parser')

            # Find article links (you may need to adjust selectors)
            article_links = soup.find_all('a', href=True)

            for link in article_links:
                href = link.get('href')
                if href and '/fake-news/' in href or '/fact-check/' in href:
                    full_url = urljoin(base_url, href)

                    # Extract article content
                    article_data = scraper.extract_article_with_newspaper3k(full_url)

                    if article_data and article_data['text']:
                        article_data['source'] = 'alt_news'
                        article_data['label'] = 'unreliable'  # Fact-checked as false
                        article_data['scraped_date'] = datetime.now().isoformat()

                        scraped_articles.append(article_data)
                        print(f"Scraped Alt News article {len(scraped_articles)}: {article_data['title'][:50]}...")

                        if len(scraped_articles) >= max_articles:
                            break

                scraper.random_delay()

            if len(scraped_articles) >= max_articles:
                break

        except Exception as e:
            print(f"Error scraping {search_url}: {e}")

    # Save to CSV
    if scraped_articles:
        df = pd.DataFrame(scraped_articles)
        output_path = "data/raw/indian_context/alt_news_articles.csv"
        df.to_csv(output_path, index=False)
        print(f"Saved {len(scraped_articles)} Alt News articles to {output_path}")

    return scraped_articles

# Execute Alt News scraping
altnews_articles = scrape_altnews_articles(max_articles=10)

In [17]:
!pip install pymongo google-generativeai tqdm

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 17.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 6.6 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [22]:
import os
import json
import pandas as pd
from pymongo import MongoClient
import google.generativeai as genai
from tqdm import tqdm

def create_enriched_csv_from_mongo(mongo_url: str, db_name: str, collection_name: str, gemini_api_key: str, output_csv_path: str, batch_size: int = 50):
    """
    Connects to MongoDB, fetches articles, enriches them with AI summaries and keywords,
    and saves the result to a CSV file.

    Args:
        mongo_url (str): The connection string for the MongoDB instance.
        db_name (str): The name of the database (e.g., 'mydb').
        collection_name (str): The name of the collection (e.g., 'scraped_articles').
        gemini_api_key (str): Your API key for the Gemini API.
        output_csv_path (str): The file path to save the final CSV.
        batch_size (int): The number of articles to process in each API call.
    """
    print("Starting the process...")

    # --- 1. Configure Gemini API ---
    try:
        genai.configure(api_key=gemini_api_key)
        model = genai.GenerativeModel('gemini-2.5-flash')
        print("Gemini API configured successfully.")
    except Exception as e:
        print(f"Error configuring Gemini API: {e}")
        return

    # --- 2. Connect to MongoDB and Fetch Data ---
    try:
        print(f"Connecting to MongoDB at {mongo_url}...")
        client = MongoClient(mongo_url)
        db = client[db_name]
        collection = db[collection_name]

        print(f"Fetching articles from '{collection_name}' collection...")
        articles_cursor = collection.find({})
        articles = list(articles_cursor)
        client.close()

        if not articles:
            print("No articles found in the collection. Exiting.")
            return

        print(f"Found {len(articles)} articles to process.")

    except Exception as e:
        print(f"Error connecting to or fetching from MongoDB: {e}")
        return

    # --- 3. Process Articles in Batches ---
    processed_data = []

    # Create batches of articles
    article_batches = [articles[i:i + batch_size] for i in range(0, len(articles), batch_size)]

    print(f"Processing articles in {len(article_batches)} batches of up to {batch_size}...")

    for batch in tqdm(article_batches, desc="Enriching Batches"):
        # Prepare the input for the Gemini API
        prompt_input = []
        for article in batch:
            prompt_input.append({
                "article_id": str(article['_id']),
                "content": article.get('content', '')[:15000] # Truncate content to avoid large prompts
            })

        # Define the prompt for Gemini
        prompt = f"""
        You are an expert news analyst. Your task is to process a JSON array of news articles.
        For each article object provided, you must:
        1.  Extract a suitable title from the beginning of the content.
        2.  Generate a concise, one-sentence summary.
        3.  Generate a list of 5 to 7 relevant keywords.

        Your response MUST be a valid JSON array where each object contains 'article_id', 'title', 'summary', and 'keywords'. Do not include any other text or explanations outside of the JSON array.

        --- EXAMPLE INPUT ---
        [
          {{
            "article_id": "example_123",
            "content": "India's space agency, ISRO, successfully launched its new communication satellite, GSAT-30, from French Guiana today. The satellite is expected to boost DTH television services, connectivity to ATMs, and stock exchange operations. ISRO chairman K. Sivan called the launch a major step forward for the country's space program."
          }}
        ]
        --- EXAMPLE OUTPUT ---
        [
          {{
            "article_id": "example_123",
            "title": "ISRO Successfully Launches GSAT-30 Communication Satellite",
            "summary": "India's space agency, ISRO, has successfully launched the GSAT-30 communication satellite to enhance DTH television services and other connectivity solutions.",
            "keywords": ["ISRO", "GSAT-30", "Satellite Launch", "Space Program", "Communication", "French Guiana", "DTH"]
          }}
        ]

        --- ACTUAL TASK ---
        Now, process the following articles:
        Input articles:
        {json.dumps(prompt_input, indent=2)}
        """

        try:
            # Call the Gemini API
            response = model.generate_content(prompt)

            # Clean and parse the JSON response
            cleaned_response = response.text.strip().replace('```json', '').replace('```', '')
            gemini_results = json.loads(cleaned_response)

            # Create a dictionary for easy lookup
            results_map = {item['article_id']: item for item in gemini_results}

            # --- 4. Map Results and Format Data ---
            for article in batch:
                article_id_str = str(article['_id'])
                ai_data = results_map.get(article_id_str)

                if ai_data:
                    # Map source to the required format
                    source_raw = article.get('source', 'unknown')
                    source_formatted = source_raw.lower().replace('the', '').replace(' ', '_')

                    processed_data.append({
                        'title': ai_data.get('title', 'N/A'),
                        'text': article.get('content', ''),
                        'authors': [],  # Schema has no author field
                        'publish_date': article.get('createdAt'), # Using createdAt as publish_date
                        'url': article.get('url', ''),
                        'source': source_formatted,
                        'label': 'reliable',  # As per the previous script's logic
                        'scraped_date': article.get('updatedAt'),
                        'summary': ai_data.get('summary', 'N/A'),
                        'keywords': ai_data.get('keywords', []),
                        'top_image': ''  # Schema has no image field
                    })
        except Exception as e:
            print(f"\nAn error occurred while processing a batch: {e}. Skipping this batch.")
            continue

    # --- 5. Save to CSV ---
    if not processed_data:
        print("No data was processed successfully. CSV file will not be created.")
        return

    print("All batches processed. Creating DataFrame...")
    df = pd.DataFrame(processed_data)

    # Ensure column order matches the desired output
    final_columns = [
        'title', 'text', 'authors', 'publish_date', 'url', 'source',
        'label', 'scraped_date', 'summary', 'keywords', 'top_image'
    ]
    df = df[final_columns]

    try:
        df.to_csv(output_csv_path, index=False, encoding='utf-8')
        print(f"Successfully saved {len(df)} enriched articles to {output_csv_path}")
    except Exception as e:
        print(f"Error saving DataFrame to CSV: {e}")


MONGO_CONNECTION_URL = os.environ.get("MONGO_URL", "xxx")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "xxx")

DB_NAME = "mydb"
COLLECTION_NAME = "scraped_articles"
OUTPUT_CSV_FILE = "enriched_indian_news.csv"

if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY":
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print("!!! WARNING: Please replace 'YOUR_GEMINI_API_KEY'    !!!")
    print("!!! with your actual Gemini API key.                 !!!")
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
else:
    # --- Execute the function ---
    create_enriched_csv_from_mongo(
        mongo_url=MONGO_CONNECTION_URL,
        db_name=DB_NAME,
        collection_name=COLLECTION_NAME,
        gemini_api_key=GEMINI_API_KEY,
        output_csv_path=OUTPUT_CSV_FILE
    )

Starting the process...
Gemini API configured successfully.
Connecting to MongoDB at xxx...
Fetching articles from 'scraped_articles' collection...
Found 1635 articles to process.
Processing articles in 33 batches of up to 50...


Enriching Batches:  85%|████████▍ | 28/33 [39:54<06:24, 76.82s/it] 


An error occurred while processing a batch: Expecting ',' delimiter: line 168 column 94 (char 16083). Skipping this batch.


Enriching Batches:  88%|████████▊ | 29/33 [41:54<05:58, 89.70s/it]


An error occurred while processing a batch: Extra data: line 304 column 1 (char 28547). Skipping this batch.


...


An error occurred while processing a batch: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 250000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 53
}
]. Skipping this batch.


Enriching Batches:  97%|█████████▋| 32/33 [43:32<00:47, 47.06s/it]


An error occurred while processing a batch: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 250000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 52
}
]. Skipping this batch.


Enriching Batches: 100%|██████████| 33/33 [44:09<00:00, 80.29s/it]


All batches processed. Creating DataFrame...
Successfully saved 1416 enriched articles to enriched_indian_news.csv


In [25]:
!pip install wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9685 sha256=2dd664ffd023cda65e7f261e24aeaacb71ee2af540f2c425c972e996cb04bfd9
  Stored in directory: /home/ayush/.cache/pip/wheels/8a/b8/04/0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [29]:
def create_sentiment_dataset():
    """Create or download sentiment analysis datasets"""

    print("Setting up sentiment analysis datasets...")

    sentiment_datasets = []

    # 1. Load Financial News Sentiment Dataset from HuggingFace
    try:
        from datasets import load_dataset

        print("Loading financial sentiment dataset...")
        fin_sentiment = load_dataset("mltrev23/financial-sentiment-analysis")

        if 'train' in fin_sentiment:
            df_fin = pd.DataFrame(fin_sentiment['train'])
            df_fin['dataset_source'] = 'financial_news'
            sentiment_datasets.append(df_fin)
            print(f"Loaded financial sentiment dataset: {len(df_fin)} samples")

    except Exception as e:
        print(f"Could not load financial sentiment dataset: {e}")

    # 2. Create Twitter Sentiment140 dataset (if available)
    try:
        import httpx

        print("Attempting to download Sentiment140 dataset...")
        dest_path = "data/raw/sentiment_analysis/sentiment140.zip"
        url = "http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip"
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        print(f"Downloading {url} → {dest_path}")

        with httpx.stream("GET", url, timeout=30.0) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(dest_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, desc=os.path.basename(dest_path), ncols=80
            ) as pbar:
                for chunk in r.iter_bytes(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))
        print("Download completed.")

        # Extract and process
        import zipfile
        with zipfile.ZipFile("data/raw/sentiment_analysis/sentiment140.zip", 'r') as zip_ref:
            zip_ref.extractall("data/raw/sentiment_analysis/")

        print("Sentiment140 dataset downloaded successfully")

    except Exception as e:
        print(f"Could not download Sentiment140: {e}")

    # 3. Create news-specific sentiment samples from our scraped data
    try:
        print("Creating news-specific sentiment samples...")

        # Load our scraped articles
        alt_news_df = pd.read_csv("data/raw/indian_context/alt_news_articles.csv")
        reliable_df = pd.read_csv("data/raw/indian_context/reliable_news_articles.csv")

        # Create sentiment labels for news articles
        news_sentiment_data = []

        # Alt News articles (often negative/sensational content)
        for _, row in alt_news_df.iterrows():
            if pd.notna(row.get('text', '')) and len(str(row['text'])) > 100:
                news_sentiment_data.append({
                    'text': str(row['text'])[:500],  # First 500 chars
                    'title': str(row['title']),
                    'sentiment': 'negative',  # Fake news often sensational
                    'source': 'alt_news',
                    'dataset_source': 'indian_news'
                })

        # Reliable news (mix of neutral and positive)
        for _, row in reliable_df.iterrows():
            if pd.notna(row.get('text', '')) and len(str(row['text'])) > 100:
                # Simple heuristic for sentiment based on keywords
                text = str(row['text']).lower()
                if any(word in text for word in ['success', 'achievement', 'growth', 'positive', 'win']):
                    sentiment = 'positive'
                elif any(word in text for word in ['crisis', 'problem', 'issue', 'concern', 'decline']):
                    sentiment = 'negative'
                else:
                    sentiment = 'neutral'

                news_sentiment_data.append({
                    'text': str(row['text'])[:500],
                    'title': str(row['title']),
                    'sentiment': sentiment,
                    'source': row['source'],
                    'dataset_source': 'indian_news'
                })

        if news_sentiment_data:
            df_news_sentiment = pd.DataFrame(news_sentiment_data)
            sentiment_datasets.append(df_news_sentiment)
            print(f"Created news sentiment dataset: {len(df_news_sentiment)} samples")

    except Exception as e:
        print(f"Error creating news sentiment data: {e}")

    # Combine all sentiment datasets
    if sentiment_datasets:
        combined_sentiment = pd.concat(sentiment_datasets, ignore_index=True)
        output_path = "data/raw/sentiment_analysis/combined_sentiment_dataset.csv"
        combined_sentiment.to_csv(output_path, index=False)

        print(f"Combined sentiment dataset saved: {output_path}")
        print(f"Total samples: {len(combined_sentiment)}")
        print("Sentiment distribution:")
        print(combined_sentiment['sentiment'].value_counts())

    return combined_sentiment

# Execute sentiment dataset creation
sentiment_data = create_sentiment_dataset()

Setting up sentiment analysis datasets...
Loading financial sentiment dataset...


Repo card metadata block was not found. Setting CardData to empty.


Loaded financial sentiment dataset: 5842 samples
Attempting to download Sentiment140 dataset...
Could not download Sentiment140: Redirect response '301 Moved Permanently' for url 'http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip'
Redirect location: 'https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/301
Creating news-specific sentiment samples...
Created news sentiment dataset: 1417 samples
Combined sentiment dataset saved: data/raw/sentiment_analysis/combined_sentiment_dataset.csv
Total samples: 7259
Sentiment distribution:
sentiment
positive    674
neutral     440
negative    303
Name: count, dtype: int64


In [30]:
def validate_datasets():
    """Validate all downloaded datasets and provide summary"""

    print("=" * 50)
    print("DATASET VALIDATION SUMMARY")
    print("=" * 50)

    datasets_info = {}

    # Check LIAR dataset
    try:
        liar_files = list(Path("data/raw/liar").glob("*.csv"))
        if liar_files:
            total_liar = sum(len(pd.read_csv(f)) for f in liar_files)
            datasets_info['LIAR'] = {
                'status': 'SUCCESS',
                'files': len(liar_files),
                'total_samples': total_liar
            }
        else:
            datasets_info['LIAR'] = {'status': 'MISSING', 'files': 0, 'total_samples': 0}
    except Exception as e:
        datasets_info['LIAR'] = {'status': f'ERROR: {e}', 'files': 0, 'total_samples': 0}

    # Check FakeNewsNet dataset
    try:
        fakenewsnet_files = list(Path("data/raw/fakenewsnet").glob("*.csv"))
        if fakenewsnet_files:
            total_fakenews = sum(len(pd.read_csv(f)) for f in fakenewsnet_files)
            datasets_info['FakeNewsNet'] = {
                'status': 'SUCCESS',
                'files': len(fakenewsnet_files),
                'total_samples': total_fakenews
            }
        else:
            datasets_info['FakeNewsNet'] = {'status': 'MISSING', 'files': 0, 'total_samples': 0}
    except Exception as e:
        datasets_info['FakeNewsNet'] = {'status': f'ERROR: {e}', 'files': 0, 'total_samples': 0}

    # Check Indian Context dataset
    try:
        indian_files = list(Path("data/raw/indian_context").glob("*.csv"))
        if indian_files:
            total_indian = sum(len(pd.read_csv(f)) for f in indian_files)
            datasets_info['Indian Context'] = {
                'status': 'SUCCESS',
                'files': len(indian_files),
                'total_samples': total_indian
            }
        else:
            datasets_info['Indian Context'] = {'status': 'MISSING', 'files': 0, 'total_samples': 0}
    except Exception as e:
        datasets_info['Indian Context'] = {'status': f'ERROR: {e}', 'files': 0, 'total_samples': 0}

    # Check Sentiment Analysis dataset
    try:
        sentiment_files = list(Path("data/raw/sentiment_analysis").glob("*.csv"))
        if sentiment_files:
            total_sentiment = sum(len(pd.read_csv(f)) for f in sentiment_files)
            datasets_info['Sentiment Analysis'] = {
                'status': 'SUCCESS',
                'files': len(sentiment_files),
                'total_samples': total_sentiment
            }
        else:
            datasets_info['Sentiment Analysis'] = {'status': 'MISSING', 'files': 0, 'total_samples': 0}
    except Exception as e:
        datasets_info['Sentiment Analysis'] = {'status': f'ERROR: {e}', 'files': 0, 'total_samples': 0}

    # Print summary
    for dataset_name, info in datasets_info.items():
        print(f"{dataset_name}:")
        print(f"  Status: {info['status']}")
        print(f"  Files: {info['files']}")
        print(f"  Samples: {info['total_samples']}")
        print()

    # Calculate totals
    total_samples = sum(info.get('total_samples', 0) for info in datasets_info.values())
    success_count = sum(1 for info in datasets_info.values() if info['status'] == 'SUCCESS')

    print(f"OVERALL SUMMARY:")
    print(f"Successfully loaded datasets: {success_count}/{len(datasets_info)}")
    print(f"Total samples collected: {total_samples:,}")

    # Save summary to file
    with open("data/dataset_summary.json", "w") as f:
        json.dump(datasets_info, f, indent=2, default=str)

    print(f"Summary saved to: data/dataset_summary.json")

# Execute validation
validate_datasets()

DATASET VALIDATION SUMMARY
LIAR:
  Status: MISSING
  Files: 0
  Samples: 0

FakeNewsNet:
  Status: SUCCESS
  Files: 4
  Samples: 23196

Indian Context:
  Status: SUCCESS
  Files: 2
  Samples: 1426

Sentiment Analysis:
  Status: SUCCESS
  Files: 1
  Samples: 7259

OVERALL SUMMARY:
Successfully loaded datasets: 3/4
Total samples collected: 31,881
Summary saved to: data/dataset_summary.json


In [31]:
def verify_installation():
    """Verify all required packages are correctly installed"""

    print("=" * 50)
    print("INSTALLATION VERIFICATION")
    print("=" * 50)

    required_packages = [
        ('torch', 'PyTorch'),
        ('transformers', 'Transformers'),
        ('sklearn', 'Scikit-learn'),
        ('pandas', 'Pandas'),
        ('numpy', 'NumPy'),
        ('spacy', 'spaCy'),
        ('nltk', 'NLTK'),
        ('requests', 'Requests'),
        ('bs4', 'Beautiful Soup'),
        ('newspaper', 'Newspaper3k')
    ]

    verification_results = {}

    for package, display_name in required_packages:
        try:
            if package == 'bs4':
                from bs4 import BeautifulSoup
                version = "Available"
            elif package == 'sklearn':
                import sklearn
                version = sklearn.__version__
            else:
                module = __import__(package)
                version = getattr(module, '__version__', 'Available')

            verification_results[display_name] = {'status': 'SUCCESS', 'version': version}
            print(f"✓ {display_name}: {version}")

        except ImportError as e:
            verification_results[display_name] = {'status': 'MISSING', 'error': str(e)}
            print(f"✗ {display_name}: MISSING - {e}")

    # Test spaCy model
    try:
        import spacy
        nlp = spacy.load("en_core_web_lg")
        verification_results['spaCy Model (en_core_web_lg)'] = {'status': 'SUCCESS', 'version': 'Available'}
        print("✓ spaCy English model: Available")
    except Exception as e:
        verification_results['spaCy Model (en_core_web_lg)'] = {'status': 'MISSING', 'error': str(e)}
        print(f"✗ spaCy English model: MISSING - {e}")

    # Test NLTK data
    try:
        import nltk
        nltk.data.find('tokenizers/punkt')
        verification_results['NLTK Data'] = {'status': 'SUCCESS', 'version': 'Available'}
        print("✓ NLTK Data: Available")
    except Exception as e:
        verification_results['NLTK Data'] = {'status': 'MISSING', 'error': str(e)}
        print(f"✗ NLTK Data: MISSING - {e}")

    # Test GPU availability (if PyTorch available)
    try:
        import torch
        gpu_available = torch.cuda.is_available()
        gpu_count = torch.cuda.device_count() if gpu_available else 0

        if gpu_available:
            gpu_info = f"Available ({gpu_count} GPU(s))"
            verification_results['GPU Support'] = {'status': 'SUCCESS', 'version': gpu_info}
            print(f"✓ GPU Support: {gpu_info}")
        else:
            verification_results['GPU Support'] = {'status': 'NO GPU', 'version': 'CPU Only'}
            print("⚠ GPU Support: Not available (CPU only)")

    except Exception:
        verification_results['GPU Support'] = {'status': 'UNKNOWN', 'version': 'Unknown'}
        print("⚠ GPU Support: Unknown")

    print("\n" + "=" * 50)

    # Count successful installations
    success_count = sum(1 for result in verification_results.values()
                       if result['status'] == 'SUCCESS')
    total_count = len([r for r in verification_results.values()
                      if r['status'] in ['SUCCESS', 'MISSING']])

    print(f"VERIFICATION SUMMARY: {success_count}/{total_count} components ready")

    # Save verification results
    with open("installation_verification.json", "w") as f:
        json.dump(verification_results, f, indent=2)

    print("Verification results saved to: installation_verification.json")

    return verification_results

# Execute verification
verify_installation()

INSTALLATION VERIFICATION
✓ PyTorch: 2.8.0+cu129
✓ Transformers: 4.55.4
✓ Scikit-learn: 1.7.1
✓ Pandas: 2.3.2
✓ NumPy: 2.1.2
✓ spaCy: 3.8.7
✓ NLTK: 3.9.1
✓ Requests: 2.32.5
✓ Beautiful Soup: Available
✓ Newspaper3k: 0.2.8
✓ spaCy English model: Available
✓ NLTK Data: Available
✓ GPU Support: Available (1 GPU(s))

VERIFICATION SUMMARY: 13/13 components ready
Verification results saved to: installation_verification.json


{'PyTorch': {'status': 'SUCCESS', 'version': '2.8.0+cu129'},
 'Transformers': {'status': 'SUCCESS', 'version': '4.55.4'},
 'Scikit-learn': {'status': 'SUCCESS', 'version': '1.7.1'},
 'Pandas': {'status': 'SUCCESS', 'version': '2.3.2'},
 'NumPy': {'status': 'SUCCESS', 'version': '2.1.2'},
 'spaCy': {'status': 'SUCCESS', 'version': '3.8.7'},
 'NLTK': {'status': 'SUCCESS', 'version': '3.9.1'},
 'Requests': {'status': 'SUCCESS', 'version': '2.32.5'},
 'Beautiful Soup': {'status': 'SUCCESS', 'version': 'Available'},
 'Newspaper3k': {'status': 'SUCCESS', 'version': '0.2.8'},
 'spaCy Model (en_core_web_lg)': {'status': 'SUCCESS', 'version': 'Available'},
 'NLTK Data': {'status': 'SUCCESS', 'version': 'Available'},
 'GPU Support': {'status': 'SUCCESS', 'version': 'Available (1 GPU(s))'}}

In [32]:
def run_quick_tests():
    """Run quick functionality tests"""

    print("=" * 50)
    print("QUICK FUNCTIONALITY TESTS")
    print("=" * 50)

    # Test 1: Basic NLP with spaCy
    try:
        import spacy
        nlp = spacy.load("en_core_web_lg")

        test_text = "This is a test sentence for fake news detection."
        doc = nlp(test_text)

        print("✓ spaCy NLP Test:")
        print(f"  Tokens: {[token.text for token in doc]}")
        print(f"  Entities: {[(ent.text, ent.label_) for ent in doc.ents]}")

    except Exception as e:
        print(f"✗ spaCy NLP Test Failed: {e}")

    # Test 2: Transformers model loading
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        test_encoding = tokenizer("This is a test", return_tensors="pt")

        print("✓ Transformers Test:")
        print(f"  Tokenizer loaded: distilbert-base-uncased")
        print(f"  Test encoding shape: {test_encoding['input_ids'].shape}")

    except Exception as e:
        print(f"✗ Transformers Test Failed: {e}")

    # Test 3: Data loading
    try:
        import pandas as pd

        # Try to load one of our datasets
        data_files = list(Path("data/raw").rglob("*.csv"))
        if data_files:
            sample_file = data_files[0]
            df = pd.read_csv(sample_file, nrows=5)  # Load first 5 rows

            print("✓ Data Loading Test:")
            print(f"  Sample file: {sample_file.name}")
            print(f"  Columns: {list(df.columns)}")
            print(f"  Sample shape: {df.shape}")
        else:
            print("⚠ No CSV files found to test data loading")

    except Exception as e:
        print(f"✗ Data Loading Test Failed: {e}")

    # Test 4: Web scraping capability
    try:
        import requests
        from bs4 import BeautifulSoup

        # Test with a simple webpage
        response = requests.get("https://httpbin.org/html", timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        print("✓ Web Scraping Test:")
        print(f"  Status code: {response.status_code}")
        print(f"  HTML parsed: {bool(soup.title)}")

    except Exception as e:
        print(f"✗ Web Scraping Test Failed: {e}")

    print("\n" + "=" * 50)
    print("PHASE 1 SETUP COMPLETED!")
    print("You can now proceed to Phase 2: Model Training")
    print("=" * 50)

# Execute quick tests
run_quick_tests()

QUICK FUNCTIONALITY TESTS
✓ spaCy NLP Test:
  Tokens: ['This', 'is', 'a', 'test', 'sentence', 'for', 'fake', 'news', 'detection', '.']
  Entities: []
✓ Transformers Test:
  Tokenizer loaded: distilbert-base-uncased
  Test encoding shape: torch.Size([1, 6])
✓ Data Loading Test:
  Sample file: combined_sentiment_dataset.csv
  Columns: ['Sentence', 'Sentiment', 'dataset_source', 'text', 'title', 'sentiment', 'source']
  Sample shape: (5, 7)
✓ Web Scraping Test:
  Status code: 200
  HTML parsed: False

PHASE 1 SETUP COMPLETED!
You can now proceed to Phase 2: Model Training
